# 🔍 Mini Semantic Search System
**AWS AASTU AI/ML Training Program — Week 2**

---

## What will we build?

A **Mini Semantic Search System** that searches FAQ documents based on **meaning**, not just keywords.

If you type: *"How does a machine learn from data?"* the system finds FAQs about
**Machine Learning** even if those exact words are not in the document.

---

## Key Concepts (read before starting)

| Concept | Simple Explanation |
|---|---|
| **Embedding** | A list of numbers representing the *meaning* of text. Similar texts get similar numbers. |
| **Semantic Search** | Finding results by *meaning*, not exact words. |
| **Pandas** | Python library for data tables (like Excel but in Python). |
| **ChromaDB** | A database that stores embeddings and finds the most similar to a query. |
| **Cosine Similarity** | Score from 0 to 1 — how similar two embeddings are. 1.0 = identical meaning. |

---

## Pipeline Overview

```
CSV File
  -> Stage 2: Pandas loads data
  -> Stage 3: Clean text
  -> Stage 4: Split into chunks
  -> Stage 5: Sentence Transformer creates embeddings
  -> Stage 6-7: ChromaDB stores everything
  -> Stage 8: User query -> embed -> ChromaDB search -> Top 3 results
```


---
## Stage 1: Install Dependencies

Run this cell first. It installs all required Python libraries.

- **pandas** — loads and cleans CSV data  
- **sentence-transformers** — the free AI model for creating embeddings  
- **chromadb** — the vector database for semantic search  

> ⏱️ This may take 2-5 minutes on first run. Wait for it to finish before continuing.


In [ ]:
# Install all required libraries
# The -q flag reduces output noise
%pip install -q pandas==2.2.2 sentence-transformers==3.0.1 chromadb==0.5.5 tqdm==4.66.4
print('Installation complete!')

In [ ]:
# Import all libraries we need
import os      # file path operations
import re      # regular expressions for text cleaning
import pandas as pd                            # data table operations
import chromadb                                # vector database
from sentence_transformers import SentenceTransformer  # embedding model

print('All libraries imported successfully!')

---
## Environment Setup

This cell detects whether you are running in **Google Colab** or **VS Code**
and sets the correct file paths automatically.


In [ ]:
try:
    import google.colab
    IN_COLAB = True
    print('Running in: Google Colab')
except ImportError:
    IN_COLAB = False
    print('Running in: VS Code / Local Jupyter')

if IN_COLAB:
    # In Colab everything goes under /content
    BASE_DIR   = '/content'
    DATA_DIR   = '/content/data'
    CSV_PATH   = '/content/data/faqs.csv'
    CHROMA_DIR = '/content/data/chroma_db'
    os.makedirs(DATA_DIR, exist_ok=True)
else:
    # In VS Code: notebooks/ is one level below the project root
    BASE_DIR   = os.path.dirname(os.getcwd())
    DATA_DIR   = os.path.join(BASE_DIR, 'data')
    CSV_PATH   = os.path.join(DATA_DIR, 'faqs.csv')
    CHROMA_DIR = os.path.join(DATA_DIR, 'chroma_db')
    os.makedirs(DATA_DIR, exist_ok=True)

print(f'CSV path  : {CSV_PATH}')
print(f'ChromaDB  : {CHROMA_DIR}')

---
## Stage 2: Load FAQ Data

**Pandas** reads our CSV file into a **DataFrame** — a table with rows and columns.

Our `faqs.csv` has 4 columns:
- `id` — row number
- `category` — topic (Python, AI, ML, AWS, Software Engineering)
- `question` — the FAQ question
- `answer` — the FAQ answer

This cell also **creates** the CSV file when running in Google Colab.


In [ ]:
# Create the FAQ CSV file
# In VS Code this file already exists in data/faqs.csv
# In Google Colab we create it here from the embedded data

faq_data = """id,category,question,answer
1,Python,What is Python?,Python is a high-level programming language with simple readable syntax. Created by Guido van Rossum in 1991 it is used in web development data science AI automation and scientific computing. Its large library and active community make it beginner-friendly.
2,Python,What is a Python list?,A list is an ordered mutable collection stored in a variable. Created with square brackets like my_list equals 1 2 3. Add items with append remove with remove and access by index starting from zero.
3,Python,What is the difference between a list and a tuple?,A list is mutable meaning you can change it after creation. A tuple is immutable and cannot be changed once created. Lists use square brackets tuples use parentheses. Use tuples for fixed data and lists for changing data.
4,Python,What are Python functions?,A function is reusable code defined with the def keyword that performs a specific task. Functions avoid code repetition and improve readability. Pass data in via parameters and get results back with the return statement.
5,Python,What is Pandas in Python?,Pandas is a Python library for data manipulation and analysis. It provides the DataFrame structure like a spreadsheet with rows and columns. Pandas loads CSV and Excel files cleans missing values filters rows and performs calculations.
6,Python,What is a Python dictionary?,A dictionary stores key-value pairs where each key is unique. Created with curly braces like person equals name Alice age 25. Access values by key name. Useful for structured data like records about a person or product.
7,Artificial Intelligence,What is Artificial Intelligence?,AI is the ability of machines to perform tasks normally requiring human intelligence such as understanding language recognizing images making decisions and learning from experience. Used in voice assistants recommendation systems self-driving cars and medical diagnosis.
8,Artificial Intelligence,What is the difference between AI and Machine Learning?,AI is the broad concept of machines simulating human intelligence. Machine Learning is a technique within AI where machines learn from data rather than following explicit rules. Think of AI as the goal and ML as the main method to achieve it.
9,Artificial Intelligence,What is a neural network?,A neural network is a computing system inspired by the human brain. It has layers of connected nodes called neurons. Data flows through layers and the model learns by adjusting connection strengths during training. Neural networks power image recognition speech and language translation.
10,Artificial Intelligence,What is Natural Language Processing?,NLP enables computers to understand interpret and generate human language. It powers chatbots search engines spam filters translation tools and sentiment analysis.
11,Artificial Intelligence,What is semantic search?,Semantic search understands the meaning behind a query rather than matching exact keywords. It uses embeddings to find conceptually similar results. Searching how to start coding might return programming tutorials even without exact word matches.
12,Machine Learning,What is Machine Learning?,Machine Learning lets computers learn patterns from data without being explicitly programmed for every scenario. You provide examples and the model learns from them. Used in spam detection recommendations fraud detection and many other fields.
13,Machine Learning,What is supervised learning?,Supervised learning trains models on labeled data where each input has a correct output. The model learns input-output relationships to predict results for new inputs. Examples include email spam classification and house price prediction.
14,Machine Learning,What is unsupervised learning?,Unsupervised learning trains on unlabeled data and finds hidden patterns independently. Clustering groups similar data points together. Customer segmentation by purchasing behavior is a common example.
15,Machine Learning,What is overfitting?,Overfitting is when a model memorizes training data including noise and performs poorly on new data. Like a student memorizing answers without understanding concepts. Prevented with more data regularization and simpler model architectures.
16,Machine Learning,What are embeddings?,Embeddings are numerical vector representations of text that capture meaning. Similar texts get vectors that are close together in vector space. King and queen are closer than king and bicycle. Embeddings power modern NLP and semantic search.
17,Machine Learning,What is a training dataset?,A training dataset is data used to teach a machine learning model patterns. The model analyzes it and adjusts to reduce prediction errors. Data quality size and diversity directly determine model performance.
18,AWS,What is Amazon Web Services?,AWS is Amazon cloud platform offering over 200 services including servers databases storage networking and machine learning tools. Build and run applications without managing physical hardware. Pay only for what you use making it cost-effective.
19,AWS,What is Amazon S3?,Amazon S3 or Simple Storage Service stores and retrieves any amount of data from the internet. Used for websites backups log files and machine learning datasets. Data is organized in containers called buckets each object identified by a unique key.
20,AWS,What is Amazon SageMaker?,SageMaker is a fully managed AWS service for machine learning. It provides tools for data labeling training hyperparameter tuning and deploying models as APIs without managing servers so data scientists focus on building models.
21,AWS,What is AWS Lambda?,Lambda is serverless computing that runs code without managing servers. Code executes on events like HTTP requests or S3 file uploads. You pay only for actual compute time making it very cost-efficient for event-driven apps.
22,Software Engineering,What is version control?,Version control tracks file changes over time for recall and team collaboration. Git is the most used system. Commits save snapshots branches allow independent work and GitHub hosts remote repositories in the cloud.
23,Software Engineering,What is a REST API?,A REST API uses HTTP methods GET POST PUT and DELETE to let systems communicate. Connects frontend applications with backend services and databases. Returns data in JSON format which is easy for both humans and machines to read.
24,Software Engineering,What is agile software development?,Agile delivers software in short sprints of one to four weeks. Teams collaborate closely adapt to requirements and review frequently. Scrum and Kanban are popular frameworks. Agile helps deliver value to users continuously.
25,Software Engineering,What is object-oriented programming?,OOP organizes code around objects that combine data attributes and behavior methods. The four principles are encapsulation inheritance polymorphism and abstraction. Makes large programs easier to design maintain and extend over time.
"""

if not os.path.exists(CSV_PATH):
    with open(CSV_PATH, 'w', encoding='utf-8') as f:
        f.write(faq_data)
    print(f'Created CSV: {CSV_PATH}')
else:
    print(f'CSV already exists: {CSV_PATH}')

In [ ]:
# Stage 2: Load the CSV into a Pandas DataFrame
df = pd.read_csv(CSV_PATH)

print(f'Loaded {len(df)} FAQ rows')
print(f'Columns : {list(df.columns)}')
print(f'Shape   : {df.shape}')
print()

# Show the first 5 rows as a formatted table
df.head()

---
## Stage 3: Inspect and Clean the Data

Real data is messy. We need to:
1. Check for **missing values** (empty cells in our CSV)
2. Remove **extra whitespace, newlines, and tabs**
3. Remove **non-printable characters**
4. Drop rows that become empty after cleaning

We use **regular expressions (regex)** — a powerful mini-language for finding
and replacing patterns in text.


In [ ]:
# Inspect the raw data before cleaning
print('=== Data Types ===')
print(df.dtypes)
print()
print('=== Missing Values per Column ===')
print(df.isnull().sum())
print()
print('=== FAQ Count by Category ===')
print(df['category'].value_counts())

In [ ]:
def clean_text(text):
    """
    Clean a single string of text.
    Steps: convert to str, strip whitespace,
    remove newlines/tabs, collapse spaces, remove non-ASCII.
    """
    if not isinstance(text, str):
        text = str(text)
    text = text.strip()                          # remove outer whitespace
    text = re.sub(r'[\n\r\t]+', ' ', text)       # newlines -> space
    text = re.sub(r' {2,}', ' ', text)           # multiple spaces -> one
    text = re.sub(r'[^\x20-\x7E]+', '', text)   # remove non-ASCII
    return text.strip()


# Drop rows missing a question or answer
df = df.dropna(subset=['question', 'answer'])

# Apply clean_text to every question and answer
# .apply() runs the function on every row automatically
df['question'] = df['question'].apply(clean_text)
df['answer']   = df['answer'].apply(clean_text)

# Drop rows where cleaning left an empty string
df = df[df['question'].str.strip() != '']
df = df[df['answer'].str.strip()   != '']
df = df.reset_index(drop=True)

print(f'Clean dataset: {len(df)} rows')
print()
print('Sample cleaned question:', df.loc[0, 'question'])
print('Sample cleaned answer  :', df.loc[0, 'answer'][:150])

---
## Stage 4: Split Long Answers into Chunks

### Why chunk?
Embedding models work best on **short texts** (under ~400 words).
Long answers are split so each section gets its own focused embedding.

### Why overlap?
Splitting at exactly 500 characters might cut a sentence in half.
**Overlap** (repeating the last 50 chars in the next chunk) prevents that.

```
Text: |--------Chunk 1---------|--------Chunk 2---------|
Overlap:                   |overlap|
```


In [ ]:
CHUNK_SIZE    = 500   # maximum characters per chunk
CHUNK_OVERLAP = 50    # characters shared between adjacent chunks


def split_text_into_chunks(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Split text into overlapping chunks."""
    chunks = []
    start  = 0
    while start < len(text):
        end   = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:               # ignore empty chunks
            chunks.append(chunk)
        start += chunk_size - overlap   # advance with overlap
    return chunks


def create_all_chunks(df):
    """Process every FAQ row and return all chunk records."""
    all_chunks = []
    for idx, row in df.iterrows():
        # Combine question + answer for full context
        combined = f"Question: {row['question']} Answer: {row['answer']}"
        chunks   = split_text_into_chunks(combined)
        for chunk_idx, chunk_text in enumerate(chunks):
            all_chunks.append({
                'chunk_id'  : f'faq_{idx}_chunk_{chunk_idx}',
                'chunk_text': chunk_text,
                'question'  : row['question'],
                'answer'    : row['answer'],
                'category'  : str(row.get('category', 'General')),
                'faq_id'    : str(row.get('id', idx)),
            })
    return all_chunks


all_chunks = create_all_chunks(df)

print(f'Total chunks created : {len(all_chunks)}')
print(f'Average per FAQ row  : {len(all_chunks) / len(df):.1f}')
print(f'First chunk text     : {all_chunks[0]["chunk_text"][:130]}...')

---
## Stage 5: Generate Embeddings

### What is an embedding?
An embedding is a list of 384 numbers representing the *meaning* of text.
Imagine a map where texts with similar meaning are placed close together.

| Text | Position on meaning-map |
|---|---|
| "dog" | near "puppy", "cat" |
| "machine learning" | near "AI", "neural network" |
| "cloud storage" | near "S3", "AWS", "backup" |

We use **`all-MiniLM-L6-v2`**: free, open-source, runs on CPU, ~90 MB download once.

> ⏱️ Wait for the progress bar to complete before continuing.


In [ ]:
MODEL_NAME = 'all-MiniLM-L6-v2'

print(f'Loading model: {MODEL_NAME}')
print('(Downloads ~90 MB on first run, then cached locally)')
print()

# Load the embedding model
model = SentenceTransformer(MODEL_NAME)
print('Model loaded!')
print()

# Extract all chunk texts as a plain list of strings
texts = [chunk['chunk_text'] for chunk in all_chunks]

print(f'Encoding {len(texts)} text chunks...')

# model.encode() converts all texts to embeddings in one call
# show_progress_bar=True displays a live % progress bar
embeddings_array = model.encode(texts, show_progress_bar=True)

# Attach each vector to its chunk record
for i, chunk in enumerate(all_chunks):
    chunk['embedding'] = embeddings_array[i].tolist()  # numpy -> Python list

print(f'\nDone! Vector size: {len(all_chunks[0]["embedding"])} dimensions')
print('First 8 numbers of embedding #0:')
print([round(x, 4) for x in all_chunks[0]['embedding'][:8]])

---
## Stage 6: Create a ChromaDB Collection

### What is ChromaDB?
ChromaDB is a **vector database** designed to store and search embeddings.

When you give ChromaDB a query vector, it finds stored vectors that are
closest in meaning using **cosine similarity**:
- Score `1.0` = perfectly identical meaning
- Score `0.5` = somewhat related
- Score `0.0` = completely unrelated

**PersistentClient** saves the database to disk — data survives between runs.


In [ ]:
COLLECTION_NAME = 'faq_collection'

# Create the storage folder if it doesn't exist
os.makedirs(CHROMA_DIR, exist_ok=True)

# Connect to (or create) the ChromaDB database on disk
client = chromadb.PersistentClient(path=CHROMA_DIR)

# get_or_create_collection: safe to call multiple times
# If collection already exists it just returns it
collection = client.get_or_create_collection(
    name     = COLLECTION_NAME,
    metadata = {'hnsw:space': 'cosine'}  # use cosine similarity
)

print(f'Collection  : {COLLECTION_NAME}')
print(f'Items stored: {collection.count()}')

---
## Stage 7: Store Embeddings and Metadata

For each chunk we store four things in ChromaDB:
- **id** — unique identifier string
- **embedding** — the 384-number vector
- **document** — the raw chunk text
- **metadata** — original question, full answer, category, faq_id

Existing IDs are checked first to prevent duplicates when re-running the notebook.


In [ ]:
# Check which IDs are already in ChromaDB
existing_ids = set(collection.get()['ids'])
print(f'Already stored: {len(existing_ids)}')

# Keep only new chunks that haven't been stored yet
new_chunks = [c for c in all_chunks if c['chunk_id'] not in existing_ids]
print(f'New to add    : {len(new_chunks)}')

if new_chunks:
    # Build the four parallel lists ChromaDB expects
    ids        = [c['chunk_id']   for c in new_chunks]
    embs       = [c['embedding']  for c in new_chunks]
    docs       = [c['chunk_text'] for c in new_chunks]
    metas      = [
        {
            'question': c['question'],
            'answer'  : c['answer'],
            'category': c['category'],
            'faq_id'  : c['faq_id'],
        }
        for c in new_chunks
    ]

    # Add data in batches of 50 to avoid memory issues
    batch_size = 50
    for start in range(0, len(ids), batch_size):
        end = start + batch_size
        collection.add(
            ids        = ids[start:end],
            embeddings = embs[start:end],
            documents  = docs[start:end],
            metadatas  = metas[start:end],
        )

    print(f'Stored {len(new_chunks)} new chunks successfully!')
else:
    print('Nothing new to store (all chunks already in ChromaDB).')

print(f'Total items in ChromaDB: {collection.count()}')

---
## Stage 8: Implement Semantic Search

### How the search works (step by step):

```
1. User types a question (text string)
2. model.encode(question) -> query vector (384 numbers)
3. ChromaDB.query(query_vector) -> finds stored vectors closest in meaning
4. Returns top 3 results with metadata and similarity scores
5. Display results to user
```

### Why does it find related meaning?
We used the **same model** to embed both FAQ texts and the user query.
So texts with similar meaning land near each other in vector space.
ChromaDB simply finds the nearest neighbours.


In [ ]:
TOP_K = 3  # return top 3 results


def semantic_search(query, collection, model, top_k=TOP_K):
    """
    Find the top-k most relevant FAQ answers for a query.

    Steps:
      1. Embed the query using the same model
      2. Ask ChromaDB for nearest stored vectors
      3. Convert distances to similarity scores
      4. Return formatted result list
    """
    if not query.strip():
        print('Please enter a question.')
        return []

    # Step 1: embed the user query
    query_vector = model.encode(query).tolist()

    # Step 2: query ChromaDB for nearest neighbours
    raw = collection.query(
        query_embeddings = [query_vector],
        n_results        = top_k,
        include          = ['metadatas', 'documents', 'distances']
    )

    # Step 3: format the results
    results = []
    for rank, (meta, dist, doc) in enumerate(
            zip(raw['metadatas'][0], raw['distances'][0], raw['documents'][0]),
            start=1):

        # cosine distance 0=identical, 1=opposite -> convert to similarity
        similarity = round(1.0 - dist, 4)

        results.append({
            'rank'            : rank,
            'question'        : meta.get('question', 'N/A'),
            'answer'          : meta.get('answer',   'N/A'),
            'category'        : meta.get('category', 'N/A'),
            'similarity_score': similarity,
        })

    return results


def display_results(query, results):
    """Print search results in a clear readable format."""
    print()
    print('=' * 60)
    print(f'  QUERY: {query}')
    print('=' * 60)

    if not results:
        print('  No results found.')
        return

    for r in results:
        filled = int(r['similarity_score'] * 20)
        bar    = chr(9608) * filled + chr(9617) * (20 - filled)
        answer = r['answer'][:280] + ('...' if len(r['answer']) > 280 else '')

        print(f"\n  Rank #{r['rank']}  |  Category: {r['category']}")
        print(f"  Similarity : {r['similarity_score']:.4f}  [{bar}]")
        print(f"  Question   : {r['question']}")
        print(f"  Answer     : {answer}")
        print('  ' + '-' * 55)

    print()


print('Semantic search functions are ready!')

---
## Stage 9: Test with 5 Example Queries

These queries don't use the exact words from the FAQs.
Semantic search finds the right answers **by meaning**.

Each query will show the **top 3 most similar FAQ answers**.


In [ ]:
# Stage 9: Run 5 test queries

test_queries = [
    'How does machine learning actually work?',
    'What is a word embedding in AI?',
    'How do I store files in the AWS cloud?',
    'How do teams share and track code changes?',
    'How do I store key-value data in Python?',
]

print('Running 5 test queries. Each shows TOP 3 results.')
print()

for query in test_queries:
    results = semantic_search(query, collection, model)
    display_results(query, results)

---
## Stage 10: Ask Your Own Question

Change the `my_question` string below to **any question you want**.
Run the cell to see the top 3 most relevant FAQ answers.

> Ideas to try:
> - *"What is overfitting?"*
> - *"Tell me about serverless computing on AWS"*
> - *"How do agile teams work?"*
> - *"What tools help analyse tabular data in Python?"*


In [ ]:
# Change this question to anything you want to search!
my_question = 'What is the difference between supervised and unsupervised learning?'

results = semantic_search(my_question, collection, model)
display_results(my_question, results)

In [ ]:
# Try a few more queries to explore the system
extra_queries = [
    'What AWS service is good for machine learning workflows?',
    'How does a search engine understand meaning in text?',
    'What are the principles of object-oriented design?',
]

for q in extra_queries:
    results = semantic_search(q, collection, model)
    display_results(q, results)

---
## Summary

Congratulations! You built a complete **Mini Semantic Search System**.

| Stage | What We Did | Tool |
|---|---|---|
| 1 | Install dependencies | pip |
| 2 | Load FAQ CSV data | Pandas |
| 3 | Clean text (spaces, chars) | Python regex |
| 4 | Split answers into chunks | Custom Python function |
| 5 | Convert text to embeddings | Sentence Transformers |
| 6 | Create vector database | ChromaDB |
| 7 | Store embeddings + metadata | ChromaDB |
| 8 | Search by meaning | ChromaDB + Sentence Transformers |
| 9 | Run 5 test queries | Our search function |
| 10 | Custom user queries | Interactive cells |

### Key Takeaways
1. **Semantic search** finds results by *meaning*, not exact keywords
2. **Embeddings** are the numbers that represent meaning mathematically
3. The **same model** must embed both FAQs and queries for search to work
4. **ChromaDB** makes nearest-neighbour vector search fast and simple
5. **Pandas** makes loading and cleaning tabular data straightforward
6. This entire system uses **free, open-source tools** — no API keys needed

---
*AWS AASTU AI/ML Training Program — Week 2 | Mini Semantic Search System*
